# Synthetic Dataset Generator for Solar-Powered IoT Nodes

This notebook generates a synthetic dataset to evaluate design configurations of solar-powered IoT nodes.  
Simplified model includes: 

- Real hourly irradiance and temperature data (8760 hours).
- Photovoltaic panels of multiple realistic sizes (1–400 cm²).
- Battery capacities expressed in mAh, typical for IoT devices.
- Temperature losses in the PV module, PMU conversion losses, battery charge/discharge efficiency, and minimum allowed SOC.
- Hour-by-hour simulation of the battery State of Charge (SoC) for an entire year.
- Aggregated viability metrics per configuration.

The goal is to identify panel–battery combinations that provide sufficient energy autonomy under realistic environmental conditions.


## Workflow

### **C1 setup_constants**
Defines:
- Node power consumption.
- PV panel performance parameters (η_STC, γ, NOCT).
- PMU efficiency.
- Battery constants (nominal voltage, charge/discharge efficiency, SOC minimum).
- Realistic list of PV panel areas (1–400 cm²).
- Realistic list of battery capacities in mAh.

### **C2 build_design_space**
Creates all possible combinations:

(panel_area_m2 × battery_capacity_mAh × eta_PMU)

Each combination represents one independent system configuration.

### **C3 load_irradiance_data**
Loads the file `irradiance.csv` containing:

Month, Day, Hour, G_h, T_amb

The simulation does not use actual dates or years; it assumes a continuous sequence of hours.

### **C4 compute_pv_power**
For each panel size and hour, computes:
- PV module temperature using the NOCT model.
- Temperature-corrected PV efficiency.
- PV electrical power output (`P_PV`).

Produces a long-format dataset (`df_pv`) with one row per:

hour × panel_area_m2

### **C5 compute_hourly_balance**
Computes:
- PMU-adjusted power (`P_PMU`).
- Net power balance:

P_BAT = P_PMU − NODE_POWER_W

- Net charging/discharging current using the battery voltage:

I_BAT_mA = (P_BAT / BATTERY_VOLTAGE) × 1000

### **C6 simulate_battery_soc**
Simulates the battery SoC hour by hour for every configuration:
- Iterates over all (panel × battery) combinations.
- Integrates the net current in mAh.
- Applies charge/discharge efficiency.
- Enforces SOC bounds: `SOC_MIN ≤ SoC ≤ 1`.

The resulting dataset `df_soc` contains:

hour_index, panel_area_m2, battery_capacity_mAh, SoC, I_BAT_mA, ...

### **C7 evaluate_viability**
Computes aggregated performance metrics per configuration, including:
- Fraction of time at SOC minimum (`soc_min_fraction`)
- Fraction of time at full SOC (`soc_full_fraction`)
- Surplus and deficit current-hours (`surplus_mAh`, `deficit_mAh`)
- Net energy balance (`net_mAh`)
- **Longest continuous autonomy** before reaching SOC_MIN (`autonomy_hours`)
- Mean and standard deviation of SoC over the year (`soc_mean`, `soc_std`)

The result is the table `summary`, used for interpretation and visual analysis.

## Input Files

- **irradiance.csv**  
  Hourly irradiance (Gh) and temperature (Tamb) data used for simulation.

## Final Notes

This pipeline is designed to:
- Explore a wide configuration space efficiently.
- Support future extensions (panel tilt, variable loads, PMU models, etc.).
- Remain scalable thanks to long-format data representation.



In [16]:
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import plotly

## Column definitions 

### Dataset `df_pv`
| Column name       | Unit | Description |
|-------------------|------|-------------|
| `hour_index`      | —    | Sequential hour index starting at 0 (0–8759) |
| `Month`           | —    | Calendar month from input dataset |
| `Day`             | —    | Calendar day from input dataset |
| `Hour`            | —    | Hour of day (0–23) from input dataset |
| `panel_area_m2`   | m²   | Effective PV panel area for this record |
| `G_h`             | W/m² | Global horizontal irradiance |
| `T_amb`           | °C   | Ambient temperature |
| `T_mod`           | °C   | PV module temperature (NOCT model) |
| `eta_PV`          | —    | Temperature-corrected PV efficiency |
| `P_PV`            | W    | PV power before PMU |

### Dataset `df_pv_pmu`
| Column name       | Unit | Description |
|-------------------|------|-------------|
| `hour_index`      | —    | Sequential hour index |
| `panel_area_m2`   | m²   | PV panel area |
| `eta_PMU`         | —    | PMU efficiency value |
| `P_PMU`           | W    | PV power after PMU losses (`P_PV × η_PMU`) |
| `P_BAT`           | W    | Net power flowing into the battery (`P_PMU − NODE_POWER_W`) |
| `I_BAT_mA`        | mA   | Net battery current (charging/discharging) |
| `G_h`             | W/m² | Global horizontal irradiance |
| `T_amb`           | °C   | Ambient temperature |
| `T_mod`           | °C   | PV module temperature |
| `eta_PV`          | —    | PV efficiency |
| `P_PV`            | W    | PV power before PMU |

### Dataset `df_soc`
| Column name       | Unit | Description |
|-------------------|------|-------------|
| `hour_index`      | —    | Sequential hour index |
| `panel_area_m2`   | m²   | PV panel area |
| `eta_PMU`         | —    | PMU efficiency |
| `C_batt_mAh`      | mAh  | Battery nominal capacity |
| `I_BAT_mA`        | mA   | Net current into battery |
| `SoC`             | —    | State of charge (0–1) |
| `G_h`             | W/m² | Global horizontal irradiance |
| `T_amb`           | °C   | Ambient temperature |
| `T_mod`           | °C   | PV module temperature |
| `eta_PV`          | —    | PV efficiency |
| `P_PV`            | W    | PV power before PMU |
| `P_PMU`           | W    | Power after PMU |
| `P_BAT`           | W    | Net battery power |

### Dataset `summary`
| Column name            | Unit | Description |
|------------------------|------|-------------|
| `panel_area_m2`        | m²   | PV panel area |
| `C_batt_mAh`           | mAh  | Battery capacity |
| `eta_PMU`              | —    | PMU efficiency |
| `hours_total`          | h    | Total simulated hours |
| `hours_soc_min`        | h    | Hours at or below SOC_MIN |
| `hours_soc_full`       | h    | Hours at full charge |
| `soc_mean`             | —    | Mean state of charge |
| `soc_std`              | —    | Standard deviation of SoC |
| `surplus_mAh`          | mAh  | Total surplus current-hours |
| `deficit_mAh`          | mAh  | Total deficit current-hours |
| `net_mAh`              | mAh  | Surplus minus deficit |
| `autonomy_hours`       | h    | Longest continuous run above SOC_MIN |
| `soc_min_fraction`     | —    | Fraction of time at minimum SoC |
| `soc_full_fraction`    | —    | Fraction of time at full charge |



## C1. basic_config

In [17]:
# --- Node parameters ---
NODE_POWER_W = 0.05  # Constant node power consumption (50 mW)

# --- Photovoltaic panel constants ---
ETA_STC = 0.175        # Efficiency at STC
GAMMA_PER_C = -0.0045  # Temperature coefficient (%/°C)
NOCT_C = 45.0          # Nominal Operating Cell Temperature (°C)

# --- Battery constants ---
BATTERY_ETA_C = 0.95  # Charge/discharge efficiency
SOC_MIN = 0.2         # Minimum allowed SoC (fraction)
BATTERY_VOLTAGE = 3.7  # Nominal voltage (V)

# --- Variable parameter ranges ---

pmu_eta_values = [0.87, 0.90, 0.95, 0.98]

# Photovoltaic panel areas (1–400 cm²)
panel_areas_m2 = [
    0.0001,   # 1 cm²
    0.00025,  # 2.5 cm²
    0.0004,   # 4 cm²
    0.000625, # 6.25 cm² (≈2.5×2.5 cm)
    0.0010,   # 10 cm²
    0.0025,   # 25 cm²
    0.0040,   # 40 cm²
    0.00625,  # 62.5 cm² (≈8×8 cm cell)
    0.0080,   # 80 cm²
    0.0100,   # 100 cm² (≈10×10 cm)
    0.0160,   # 160 cm² (≈12.6×12.6 cm)
    0.0250,   # 250 cm² (≈15.8×15.8 cm)
    0.0310,   # 310 cm² (≈17.6×17.6 cm)
    0.0400    # 400 cm² (≈20×20 cm)
]

# Battery capacities (realistic IoT battery sizes)
battery_capacities_mAh = [
    30,    # small Li-ion coin cell (~0.1 Wh @ 3.7 V)
    70,    # supercap or very small LiPo (~0.25 Wh @ 3.7 V)
    135,   # compact LiPo (~0.5 Wh @ 3.7 V)
    270,   # small pouch cell (~1.0 Wh @ 3.7 V)
    500,   # standard Li-ion (~2.0 Wh @ 3.7 V)
    1000,  # one 18650 cell (~3.7 Wh)
    1300,  # small LiPo pack (~5.0 Wh @ 3.7 V)
    2000,  # two small Li-ion cells (~7.4 Wh @ 3.7 V)
    2600,  # high-capacity 18650 (~10 Wh)
    4000,  # larger LiPo pack (~15 Wh)
    5400   # large IoT/multi-day autonomy pack (~20 Wh)
]


## C2. build_design_space

In [18]:
# --- Generate design space: combinations of variable parameters ---
design_space = list(itertools.product(
    panel_areas_m2,
    battery_capacities_mAh,
    pmu_eta_values
))

df_design = pd.DataFrame(design_space, columns=[
    "panel_area_m2",
    "battery_capacity_mAh",
    "eta_PMU"
])

df_design

,panel_area_m2,battery_capacity_mAh,eta_PMU
0,0.0001,30,0.87
1,0.0001,30,0.90
2,0.0001,30,0.95
3,0.0001,30,0.98
4,0.0001,70,0.87
...,...,...,...
611,0.0400,4000,0.98
612,0.0400,5400,0.87
613,0.0400,5400,0.90
614,0.0400,5400,0.95


## C3. load_irradiance_data

In [19]:
# --- Load irradiance and temperature data ---
irr_data = pd.read_csv("irradiance.csv")

# --- Basic sanity checks ---
expected_cols = ["Date-hour", "Month", "Day", "Hour", "G(h)", "Temperature"]
missing_cols = [c for c in expected_cols if c not in irr_data.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

# --- Rename columns for internal consistency ---
irr_data = irr_data.rename(columns={
    "Date-hour": "datetime_str",
    "G(h)": "G_h",
    "Temperature": "T_amb"
})

# --- Keep only relevant numeric columns ---
irr_data = irr_data[["Month", "Day", "Hour", "G_h", "T_amb"]].reset_index(drop=True)


irr_data

,Month,Day,Hour,G_h,T_amb
0,1,1,5,0,2.81
1,1,1,6,0,2.28
2,1,1,7,0,1.74
3,1,1,8,61,2.17
4,1,1,9,222,2.45
...,...,...,...,...,...
5470,12,31,15,222,10.36
5471,12,31,16,75,9.73
5472,12,31,17,0,9.19
5473,12,31,18,0,8.66


## C4. compute_pv_power

In [20]:
# --- Build a long-format DataFrame: one row per hour and panel area ---

# Duplicate irradiance data for each panel area
df_pv = pd.concat(
    [irr_data.assign(panel_area_m2=A) for A in panel_areas_m2],
    ignore_index=True
)

# Compute module temperature based on NOCT model
df_pv["T_mod"] = df_pv["T_amb"] + (NOCT_C - 20.0) / 800.0 * df_pv["G_h"]

# Compute temperature-corrected panel efficiency
df_pv["eta_PV"] = ETA_STC * (1.0 + GAMMA_PER_C * (df_pv["T_mod"] - 25.0))

# Compute PV power output
df_pv["P_PV"] = df_pv["G_h"] * df_pv["panel_area_m2"] * df_pv["eta_PV"]

# Add hour index (0..n-1)
df_pv["hour_index"] = df_pv.groupby("panel_area_m2").cumcount()

# Keep relevant columns
df_pv = df_pv[[
    "hour_index",
    "Month",
    "Day",
    "Hour",
    "panel_area_m2",
    "G_h",
    "T_amb",
    "T_mod",
    "eta_PV",
    "P_PV"
]]

df_pv

,hour_index,Month,Day,Hour,panel_area_m2,G_h,T_amb,T_mod,eta_PV,P_PV
0,0,1,1,5,0.0001,0,2.81,2.81000,0.192475,0.000000
1,1,1,1,6,0.0001,0,2.28,2.28000,0.192892,0.000000
2,2,1,1,7,0.0001,0,1.74,1.74000,0.193317,0.000000
3,3,1,1,8,0.0001,61,2.17,4.07625,0.191477,0.001168
4,4,1,1,9,0.0001,222,2.45,9.38750,0.187295,0.004158
...,...,...,...,...,...,...,...,...,...,...
76645,5470,12,31,15,0.0400,222,10.36,17.29750,0.181066,1.607864
76646,5471,12,31,16,0.0400,75,9.73,12.07375,0.185179,0.555538
76647,5472,12,31,17,0.0400,0,9.19,9.19000,0.187450,0.000000
76648,5473,12,31,18,0.0400,0,8.66,8.66000,0.187868,0.000000


## C5. compute_hourly_balance

In [21]:
# Expand df_pv for all PMU efficiencies
df_pv_pmu = pd.concat(
    [df_pv.assign(eta_PMU=eta) for eta in pmu_eta_values],
    ignore_index=True
)

# Compute PMU-adjusted power
df_pv_pmu["P_PMU"] = df_pv_pmu["P_PV"] * df_pv_pmu["eta_PMU"]

# Net power (W)
df_pv_pmu["P_BAT"] = df_pv_pmu["P_PMU"] - NODE_POWER_W

# Convert net power to net current (mA)
df_pv_pmu["I_BAT_mA"] = (df_pv_pmu["P_BAT"] / BATTERY_VOLTAGE) * 1000

df_pv_pmu


,hour_index,Month,Day,Hour,panel_area_m2,G_h,T_amb,T_mod,eta_PV,P_PV,eta_PMU,P_PMU,P_BAT,I_BAT_mA
0,0,1,1,5,0.0001,0,2.81,2.81000,0.192475,0.000000,0.87,0.000000,-0.050000,-13.513514
1,1,1,1,6,0.0001,0,2.28,2.28000,0.192892,0.000000,0.87,0.000000,-0.050000,-13.513514
2,2,1,1,7,0.0001,0,1.74,1.74000,0.193317,0.000000,0.87,0.000000,-0.050000,-13.513514
3,3,1,1,8,0.0001,61,2.17,4.07625,0.191477,0.001168,0.87,0.001016,-0.048984,-13.238873
4,4,1,1,9,0.0001,222,2.45,9.38750,0.187295,0.004158,0.87,0.003617,-0.046383,-12.535834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306595,5470,12,31,15,0.0400,222,10.36,17.29750,0.181066,1.607864,0.98,1.575706,1.525706,412.353057
306596,5471,12,31,16,0.0400,75,9.73,12.07375,0.185179,0.555538,0.98,0.544428,0.494428,133.629054
306597,5472,12,31,17,0.0400,0,9.19,9.19000,0.187450,0.000000,0.98,0.000000,-0.050000,-13.513514
306598,5473,12,31,18,0.0400,0,8.66,8.66000,0.187868,0.000000,0.98,0.000000,-0.050000,-13.513514


## C6. simulate_battery_soc

In [22]:
# --- Expand df_pv_pmu for all battery capacities ---
df_soc = pd.concat(
    [df_pv_pmu.assign(C_batt_mAh=Cbat) for Cbat in battery_capacities_mAh],
    ignore_index=True
)

df_soc["SoC"] = np.nan

# --- Simulate SoC for each configuration ---
# One configuration = (panel_area_m2, C_batt_mAh, eta_PMU)
for (A, Cbat, eta), group_idx in df_soc.groupby(
    ["panel_area_m2", "C_batt_mAh", "eta_PMU"]
).groups.items():

    idx = list(group_idx)
    i_bat = df_soc.loc[idx, "I_BAT_mA"].to_numpy()

    soc = np.empty_like(i_bat)
    soc[0] = 1.0  # start fully charged

    for i in range(1, len(i_bat)):
        delta = i_bat[i]

        if delta >= 0:
            # Charging: apply charge efficiency
            soc[i] = soc[i-1] + (delta / Cbat) * BATTERY_ETA_C
        else:
            # Discharging: apply discharge efficiency
            soc[i] = soc[i-1] + (delta / Cbat) / BATTERY_ETA_C

        # Clamp SoC to allowed range
        soc[i] = min(1.0, max(SOC_MIN, soc[i]))

    df_soc.loc[idx, "SoC"] = soc

df_soc

,hour_index,Month,Day,Hour,panel_area_m2,G_h,T_amb,T_mod,eta_PV,P_PV,eta_PMU,P_PMU,P_BAT,I_BAT_mA,C_batt_mAh,SoC
0,0,1,1,5,0.0001,0,2.81,2.81000,0.192475,0.000000,0.87,0.000000,-0.050000,-13.513514,30,1.000000
1,1,1,1,6,0.0001,0,2.28,2.28000,0.192892,0.000000,0.87,0.000000,-0.050000,-13.513514,30,0.525842
2,2,1,1,7,0.0001,0,1.74,1.74000,0.193317,0.000000,0.87,0.000000,-0.050000,-13.513514,30,0.200000
3,3,1,1,8,0.0001,61,2.17,4.07625,0.191477,0.001168,0.87,0.001016,-0.048984,-13.238873,30,0.200000
4,4,1,1,9,0.0001,222,2.45,9.38750,0.187295,0.004158,0.87,0.003617,-0.046383,-12.535834,30,0.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3372595,5470,12,31,15,0.0400,222,10.36,17.29750,0.181066,1.607864,0.98,1.575706,1.525706,412.353057,5400,1.000000
3372596,5471,12,31,16,0.0400,75,9.73,12.07375,0.185179,0.555538,0.98,0.544428,0.494428,133.629054,5400,1.000000
3372597,5472,12,31,17,0.0400,0,9.19,9.19000,0.187450,0.000000,0.98,0.000000,-0.050000,-13.513514,5400,0.997366
3372598,5473,12,31,18,0.0400,0,8.66,8.66000,0.187868,0.000000,0.98,0.000000,-0.050000,-13.513514,5400,0.994732


## C7. evaluate_viability

In [23]:
# --- Helper: compute longest autonomy interval ---
def longest_autonomy_hours(soc_series, soc_min):
    """
    Returns the longest continuous interval (in hours) during which
    SoC stays strictly above soc_min. This estimates how long the
    system can sustain operation without new solar input.
    """
    below = soc_series <= soc_min + 1e-6
    if np.all(below):
        return 0

    max_len = 0
    current = 0
    for v in below:
        if not v:
            current += 1
            max_len = max(max_len, current)
        else:
            current = 0
    return max_len


# --- Compute failure hours at the df_soc level ---
# A "failure hour" is when:
#   (1) SoC == SOC_MIN  → battery cannot discharge further
#   (2) I_BAT_mA < 0     → the node still requires power
df_soc["failure_hour"] = (
    (df_soc["SoC"] <= SOC_MIN + 1e-9) &
    (df_soc["I_BAT_mA"] < 0)
).astype(int)


# --- Aggregate metrics per configuration ---
summary = (
    df_soc
    .groupby(["panel_area_m2", "C_batt_mAh", "eta_PMU"], as_index=False)
    .agg(
        hours_total=("SoC", "count"),                      # number of hours simulated (usually 8760)
        hours_soc_min=("SoC", lambda s: np.sum(s <= SOC_MIN + 1e-6)),  # hours at or below SOC_MIN
        hours_soc_full=("SoC", lambda s: np.sum(s >= 1.0 - 1e-6)),     # hours at full capacity
        soc_mean=("SoC", "mean"),                          # mean SoC over the year
        soc_std=("SoC", "std"),                            # standard deviation of SoC
        surplus_mAh=("I_BAT_mA", lambda s: np.sum(np.clip(s, 0, None))),        # total surplus (charging)
        deficit_mAh=("I_BAT_mA", lambda s: -np.sum(np.clip(s, None, 0))),       # total deficit (discharging)
        autonomy_hours=("SoC", lambda s: longest_autonomy_hours(s.to_numpy(), SOC_MIN)),
        failure_hours=("failure_hour", "sum")              # hours where the node cannot operate
    )
)

# --- Derived metrics ---
summary["soc_min_fraction"] = summary["hours_soc_min"] / summary["hours_total"]
summary["soc_full_fraction"] = summary["hours_soc_full"] / summary["hours_total"]
summary["net_mAh"] = summary["surplus_mAh"] - summary["deficit_mAh"]

summary.head()


,panel_area_m2,C_batt_mAh,eta_PMU,hours_total,hours_soc_min,hours_soc_full,soc_mean,soc_std,surplus_mAh,deficit_mAh,autonomy_hours,failure_hours,soc_min_fraction,soc_full_fraction,net_mAh
0,0.0001,30,0.87,5475,5473,1,0.200206,0.011673,0.0,67766.361231,2,5473,0.999635,0.000183,-67766.361231
1,0.0001,30,0.90,5475,5473,1,0.200206,0.011673,0.0,67551.874153,2,5473,0.999635,0.000183,-67551.874153
2,0.0001,30,0.95,5475,5473,1,0.200206,0.011673,0.0,67194.395690,2,5473,0.999635,0.000183,-67194.395690
3,0.0001,30,0.98,5475,5473,1,0.200206,0.011673,0.0,66979.908613,2,5473,0.999635,0.000183,-66979.908613
4,0.0001,70,0.87,5475,5470,1,0.200364,0.014733,0.0,67766.361231,5,5470,0.999087,0.000183,-67766.361231


## C8. compute_optimal_score

In [24]:
# --- Multi-criteria scoring function ---
# We combine:
#   (1) battery size (minimize)
#   (2) panel size (minimize)
#   (3) autonomy (maximize)
#   (4) failure hours (minimize)
#
# Default weights give all criteria equal importance:
#   w_batt = w_panel = w_auto = w_fail = 1.0
#
# All components are normalized to [0,1] inside the function.


def compute_score(row,
                  w_batt=1.0,
                  w_panel=1.0,
                  w_auto=1.0,
                  w_fail=1.0):
    """
    Computes a composite score for a configuration based on:
      - battery capacity      (minimize)
      - panel area           (minimize)
      - autonomy             (maximize)
      - failure hours        (minimize)
    Lower score = better configuration.
    """

    # Normalize battery capacity (smaller is better)
    batt_norm = (row["C_batt_mAh"] - summary["C_batt_mAh"].min()) / \
                (summary["C_batt_mAh"].max() - summary["C_batt_mAh"].min())

    # Normalize panel area (smaller is better)
    panel_norm = (row["panel_area_m2"] - summary["panel_area_m2"].min()) / \
                 (summary["panel_area_m2"].max() - summary["panel_area_m2"].min())

    # Normalize autonomy (higher is better → subtract)
    auto_norm = (row["autonomy_hours"] - summary["autonomy_hours"].min()) / \
                (summary["autonomy_hours"].max() - summary["autonomy_hours"].min())

    # Normalize failure hours (smaller is better)
    fail_norm = (row["failure_hours"] - summary["failure_hours"].min()) / \
                (summary["failure_hours"].max() - summary["failure_hours"].min())

    # Weighted score:
    #   minimize C_batt_mAh
    #   minimize panel_area_m2
    #   maximize autonomy
    #   minimize failure_hours
    score = (
        w_batt * batt_norm +
        w_panel * panel_norm -
        w_auto * auto_norm +
        w_fail * fail_norm
    )

    return score

summary["score"] = summary.apply(compute_score, axis=1)
summary_sorted = summary.sort_values("score", ascending=True)
summary = summary_sorted
summary

,panel_area_m2,C_batt_mAh,eta_PMU,hours_total,hours_soc_min,hours_soc_full,soc_mean,soc_std,surplus_mAh,deficit_mAh,autonomy_hours,failure_hours,soc_min_fraction,soc_full_fraction,net_mAh,score
279,0.00400,270,0.98,5475,6,2986,0.931815,0.114221,226770.936965,20494.308498,5336,6,0.001096,0.545388,206276.628467,-0.831069
282,0.00400,500,0.95,5475,0,2946,0.961807,0.065468,218342.315151,20645.169795,5475,0,0.000000,0.538082,197697.145356,-0.814732
280,0.00400,500,0.87,5475,0,2837,0.958365,0.071837,195921.440940,21102.917213,5475,0,0.000000,0.518174,174818.523727,-0.814732
281,0.00400,500,0.90,5475,0,2876,0.959748,0.069254,204319.706970,20921.700132,5475,0,0.000000,0.525297,183398.006838,-0.814732
283,0.00400,500,0.98,5475,0,2985,0.962856,0.063459,226770.936965,20494.308498,5475,0,0.000000,0.545205,206276.628467,-0.814732
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,0.00025,5400,0.87,5475,5132,1,0.224979,0.112514,0.000000,58436.173348,343,5132,0.937352,0.000183,-58436.173348,1.879148
43,0.00010,5400,0.98,5475,5154,1,0.223401,0.109223,0.000000,66979.908613,321,5154,0.941370,0.000183,-66979.908613,1.883428
42,0.00010,5400,0.95,5475,5155,1,0.223364,0.109144,0.000000,67194.395690,320,5155,0.941553,0.000183,-67194.395690,1.883793
40,0.00010,5400,0.87,5475,5156,1,0.223266,0.108937,0.000000,67766.361231,319,5156,0.941735,0.000183,-67766.361231,1.884159


## C9. filter_failure

In [25]:
summary_no_fail = (
    summary[summary["failure_hours"] == 0]
    .sort_values("score", ascending=True)
    .reset_index(drop=True)
)

summary = summary_no_fail
summary

,panel_area_m2,C_batt_mAh,eta_PMU,hours_total,hours_soc_min,hours_soc_full,soc_mean,soc_std,surplus_mAh,deficit_mAh,autonomy_hours,failure_hours,soc_min_fraction,soc_full_fraction,net_mAh,score
0,0.00400,500,0.95,5475,0,2946,0.961807,0.065468,2.183423e+05,20645.169795,5475,0,0.0,0.538082,1.976971e+05,-0.814732
1,0.00400,500,0.87,5475,0,2837,0.958365,0.071837,1.959214e+05,21102.917213,5475,0,0.0,0.518174,1.748185e+05,-0.814732
2,0.00400,500,0.90,5475,0,2876,0.959748,0.069254,2.043197e+05,20921.700132,5475,0,0.0,0.525297,1.833980e+05,-0.814732
3,0.00400,500,0.98,5475,0,2985,0.962856,0.063459,2.267709e+05,20494.308498,5475,0,0.0,0.545205,2.062766e+05,-0.814732
4,0.00625,270,0.98,5475,0,3456,0.950942,0.085241,3.826508e+05,18726.124884,5475,0,0.0,0.631233,3.639246e+05,-0.801172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,0.03100,5400,0.95,5475,0,4120,0.998368,0.003555,2.047330e+06,15767.901946,5475,0,0.0,0.752511,2.031562e+06,0.774436
290,0.04000,5400,0.90,5475,0,4185,0.998409,0.003520,2.515392e+06,15533.319919,5475,0,0.0,0.764384,2.499858e+06,1.000000
291,0.04000,5400,0.95,5475,0,4195,0.998419,0.003511,2.658330e+06,15480.184561,5475,0,0.0,0.766210,2.642850e+06,1.000000
292,0.04000,5400,0.87,5475,0,4175,0.998403,0.003525,2.429632e+06,15568.185467,5475,0,0.0,0.762557,2.414064e+06,1.000000


## C10. plot_best_configurations

In [32]:
# --- C10: interactive 3D scatter of best configurations (log-colored score) ---

import plotly.express as px

def plot_best_configurations_interactive(summary_df, top_n=50):
    """
    Interactive 3D scatter plot of the top-N best configurations.
    Color scale is log-transformed to highlight the best configurations.
    """

    # Select best configurations without failures
    best = summary_df.nsmallest(top_n, "score").copy()

    # Logarithmic transformation of the score for coloring
    # Shift so that the minimum score is positive
    best["score_shifted"] = best["score"] - best["score"].min() + 1e-6
    best["score_log"] = np.log(best["score_shifted"])

    fig = px.scatter_3d(
        best,
        x="panel_area_m2",
        y="C_batt_mAh",
        z="eta_PMU",
        color="score_log",                  # logarithmic color scale
        color_continuous_scale="Viridis",
        size_max=12,
        hover_data={
            "panel_area_m2": True,
            "C_batt_mAh": True,
            "eta_PMU": True,
            "autonomy_hours": True,
            "failure_hours": True,
            "score": True,
            "score_log": True
        },
        title=f"Top {top_n} Best Configurations"
    )

    fig.update_layout(
        width=1200,
        height=800,
        scene=dict(
            xaxis_title="Panel area (m²)",
            yaxis_title="Battery capacity (mAh)",
            zaxis_title="PMU efficiency"
        ),
        coloraxis_colorbar=dict(
            title="log(score)",
            thickness=15
        )
    )

    fig.show()


plot_best_configurations_interactive(summary, top_n=100)
